

Brookfield Marketing Chatbot — Google ADK + Streamlit on Colab
==============================================================
Uses official google-adk package (Agent, Runner, InMemorySessionService)
Exposed via Cloudflare Tunnel (no account needed, faster than ngrok)


In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================

!pip install -q \
    "google-adk" \
    "google-cloud-bigquery" \
    "google-cloud-aiplatform" \
    "google-cloud-storage>=2.18.0" \
    "streamlit" \
    "plotly" \
    "google-cloud-speech" \
    "streamlit" \
    "streamlit-mic-recorder" \
    "python-dateutil" \
    --upgrade --quiet

print("✅ Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.6 MB/s eta 0:00:00

In [ ]:
from google.cloud import speech
from google.auth import default as google_auth_default

credentials, _ = google_auth_default()
client = speech.SpeechClient(credentials=credentials)
print("✅ Speech API is accessible")

✅ Speech API is accessible


In [ ]:
# ============================================================
# CELL 2 — Authenticate
# ============================================================

from google.colab import auth as google_auth
google_auth.authenticate_user()

import os
os.environ["GOOGLE_CLOUD_PROJECT"]      = "generative-insights-poc-bi"
os.environ["GOOGLE_CLOUD_LOCATION"]     = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"   # tells ADK to use Vertex AI

print("✅ Authenticated + environment configured")

✅ Authenticated + environment configured


In [ ]:
# ============================================================
# CELL 3 — Write config.py
# ============================================================

%%writefile config.py
# ── GCP Projects ──────────────────────────────────────────
BQ_PROJECT         = "funnel-data"
BQ_BILLING_PROJECT = "funnel-data"
LLM_PROJECT        = "generative-insights-poc-bi"
VERTEX_LOCATION    = "us-central1"
DATASET_ID         = "brookfield_dashboard"
MODEL_NAME         = "gemini-2.5-pro"

# ── Table descriptions — injected into agent instructions ─
TABLE_DESCRIPTIONS = {
    "Brookfield_1H_Social": {
        "description": "Social media performance (Meta, LinkedIn).",
        "key_columns": ["Date", "Campaign_New", "Tactic", "Strategy_Type_Social",
                        "Asset_Type_Social", "Data_Source_name",
                        "Cost", "Clicks", "Impressions"],
        "filter_note": "Always filter: Data_Source_name IN ('Brookfield Brand Campaign - Standard', 'Brookfield - Ad Account - Campaign')",
    },
    "Brookfield_1H_Search": {
        "description": "Paid search (Google Ads, Bing). Brand vs NonBrand via Tactic column.",
        "key_columns": ["Date", "Campaign_New", "Tactic", "Keyword_Search",
                        "Data_Source_name", "Cost", "Clicks",
                        "Impressions", "Internal_Link_Clicks"],
        "filter_note": "Always filter: Data_Source_name IN ('2026 - Corporate Brand Campaign - Keyword', 'Brookfield Brand Campaign - Search keyword')",
    },
    "Brookfield_1H_Programmatic": {
        "description": "Programmatic/display advertising.",
        "key_columns": ["Date", "Campaign_New", "Tactic",
                        "Cost", "Clicks", "Impressions"],
        "filter_note": "",
    },
    "Brookfield_1H_Executive_Summary_": {
        "description": "Cross-channel summary (Search + Social combined).",
        "key_columns": ["Date", "Channel", "Campaign_New", "Tactic",
                        "Cost", "Clicks", "Impressions"],
        "filter_note": "",
    },
}

Overwriting config.py


The _build_instructions() function in the next cell basically build the prompt or instruction for the AI agent.
Output from the function will look something like.

`{BQ_PROJECT}.{DATASET_ID}.Brookfield_1H_Social`: Social media performance (Meta, LinkedIn).
  Columns: Date, Campaign_New, Tactic, Strategy_Type_Social, Asset_Type_Social, Data_Source_name, Cost, Clicks, Impressions
  Always filter: Data_Source_name IN ('Brookfield Brand Campaign - Standard', 'Brookfield - Ad Account - Campaign')


 `{BQ_PROJECT}.{DATASET_ID}.Brookfield_1H_Search`: Paid search (Google Ads, Bing). Brand vs NonBrand via Tactic column.
  Columns: Date, Campaign_New, Tactic, Keyword_Search, Data_Source_name, Cost, Clicks, Impressions, Internal_Link_Clicks
  Always filter: Data_Source_name IN ('2026 - Corporate Brand Campaign - Keyword', 'Brookfield Brand Campaign - Search keyword')

In [ ]:
# ============================================================
# CELL 4 — Write agent.py  (official Google ADK pattern)
# ============================================================

%%writefile agent.py
"""
Brookfield ADK Agent
====================
Uses official google-adk package:
  - Agent (LlmAgent)            — the agent definition
  - Runner                      — manages the agentic loop
  - InMemorySessionService      — conversation history per session
  - Plain Python functions       — auto-wrapped as FunctionTools by ADK

ADK automatically:
  - Generates tool schemas from function signatures + docstrings
  - Decides when to call which tool
  - Handles the tool call → result → next step loop
  - Maintains conversation state via sessions
"""

import time
import asyncio
import pandas as pd
from datetime import date
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.cloud import bigquery
from google.auth import default as google_auth_default
from config import (
    BQ_PROJECT, BQ_BILLING_PROJECT, LLM_PROJECT,
    VERTEX_LOCATION, DATASET_ID, MODEL_NAME, TABLE_DESCRIPTIONS,
)

APP_NAME = "brookfield_marketing_agent"


# ── Build system instruction ───────────────────────────────
def _build_instruction() -> str:
    table_context = "\n".join([
        f"- `{BQ_PROJECT}.{DATASET_ID}.{name}`: {info['description']}\n"
        f"  Columns: {', '.join(info['key_columns'])}\n"
        f"  {info['filter_note']}"
        for name, info in TABLE_DESCRIPTIONS.items()
    ])   #using list comprehension. The loop runs first. for loop → list of strings → join into one string.

    return f"""You are a senior performance marketing analyst assistant for Brookfield.
You have access to BigQuery tables with paid media data across Search, Social, and Programmatic.

Available tables:
{table_context}

SQL rules:
- IMPORTANT: Always call run_bq_query with just the SQL string — never wrap in Python code
- Fully qualified table names: `{BQ_PROJECT}.{DATASET_ID}.table_name`
- When asked for channel-wise or all-channel data, ALWAYS use Brookfield_1H_Executive_Summary_ table — it has all channels in one place. Never query individual channel tables separately for cross-channel questions.
- Only query individual tables (Social, Search, Programmatic) when asked specifically about one channel.
- Date column is `Date` (capital D), use BETWEEN for ranges
- Use SAFE_DIVIDE() for ratios, IFNULL(col, 0) for aggregations
- Apply Data_Source_name filters as noted above
- Aggregate data — don't return raw keyword-level rows for summary questions
- For period comparisons use FULL JOIN between curr and prev CTEs

Response rules:
- Use only numbers from query results — never hallucinate
- Format: Cost: ↑18% ($5,866 vs $4,970) with ↑↓ arrows
- <5% = stable/flat | 5-15% = slightly improved/softened | >15% = sharply increased/decreased
- Never mention SQL, tables, or databases in your final answer
- If asked for a chart, add [CHART_REQUESTED] at the end of your response.
- Only add the chart if asked or you feel it's required.
- If it's a metric like CPM,CPC then increased is bad and decreased is good in comparison. As we have to spend less money for more clicks or impressions.

"""


# ── Tools — plain Python functions, ADK auto-wraps these ──
# ADK reads the function name, docstring, type hints, and
# parameter names to generate the tool schema automatically.

def get_today_date() -> str:
    """
    Returns today's date as a string in YYYY-MM-DD format.
    Always call this before computing date ranges like 'last month',
    'last quarter', 'year to date', 'last 30 days', etc.
    """
    return str(date.today())


def get_schema() -> str:
    """
    Returns the schema and column descriptions for all available
    BigQuery tables. Call this when you need to understand what
    data is available before writing a SQL query.
    """
    lines = [f"Dataset: `{BQ_PROJECT}.{DATASET_ID}`\n"]
    for table_name, info in TABLE_DESCRIPTIONS.items():
        lines.append(f"Table: `{BQ_PROJECT}.{DATASET_ID}.{table_name}`")
        lines.append(f"  {info['description']}")
        lines.append(f"  Columns: {', '.join(info['key_columns'])}")
        if info["filter_note"]:
            lines.append(f"  Filter required: {info['filter_note']}")
        lines.append("")
    return "\n".join(lines)


def run_bq_query(sql: str) -> str:
    """
    Executes a BigQuery SQL query and returns the results as a
    formatted text table. Use this to fetch data needed to answer
    the user's question. Write complete, valid BigQuery Standard SQL
    with fully qualified table names.

    Args:
        sql: A complete, valid BigQuery Standard SQL query string.

    Returns:
        Query results as a pipe-separated text table, or an error message.
    """
    try:
        credentials, _ = google_auth_default()
        client  = bigquery.Client(
            project=BQ_BILLING_PROJECT,
            credentials=credentials,
        )
        t0      = time.time()
        df      = client.query(sql).result().to_dataframe(
            create_bqstorage_client=False
        )
        elapsed = time.time() - t0

        if df.empty:
            return "Query returned no results."

        # Round numerics, fill NAs
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].round(2).fillna(0)
        df = df.fillna("—")

        # Pipe-separated table (max 100 rows to LLM)
        display_df = df.head(100)
        col_widths = {
            col: max(len(str(col)), display_df[col].astype(str).str.len().max())
            for col in display_df.columns
        }
        header    = " | ".join(str(c).ljust(col_widths[c]) for c in display_df.columns)
        separator = "-+-".join("-" * col_widths[c] for c in display_df.columns)
        rows      = [
            " | ".join(
                str(v).ljust(col_widths[c])
                for c, v in zip(display_df.columns, row)
            )
            for row in display_df.itertuples(index=False)
        ]
        text = "\n".join([header, separator] + rows)
        if len(df) > 100:
            text += f"\n... ({len(df)} rows total, showing 100)"

        return f"[{len(df)} rows returned in {elapsed:.1f}s]\n\n{text}"

    except Exception as e:
        print(f"BQ ERROR: {e}")
        return f"Query failed: {str(e)}"


# ── ADK Agent definition ───────────────────────────────────
# Tools are passed as plain functions — ADK wraps them automatically

root_agent = Agent(
    name="brookfield_analyst",
    model=MODEL_NAME,
    description="Senior performance marketing analyst for Brookfield. Queries BigQuery and generates insights.",
    instruction=_build_instruction(),
    tools=[
        get_today_date,   # ADK auto-wraps as FunctionTool
        get_schema,
        run_bq_query,
    ],
)


# ── Session service — manages conversation history ─────────
session_service = InMemorySessionService()


# ── Runner — manages the agentic loop ──────────────────────
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


# ── Helper: sync wrapper for Streamlit ─────────────────────
# ADK uses async — Streamlit is sync. This bridges the gap.

def create_session(user_id: str, session_id: str):
    """Creates a new ADK session for a user."""
    async def _create():
        await session_service.create_session(
            app_name=APP_NAME,
            user_id=user_id,
            session_id=session_id,
        )
    asyncio.run(_create())


def chat(
    user_message: str,
    user_id: str,
    session_id: str,
) -> dict:
    """
    Sends a message to the ADK agent and returns the response.

    ADK's Runner handles the full agentic loop:
      1. Sends message to Gemini
      2. If Gemini calls a tool → executes it → sends result back
      3. Repeats until Gemini produces a final text response
      4. Returns all events (tool calls, responses, etc.)

    Returns:
        {
          "text":       str,           final response text
          "tool_calls": list[str],     names of tools called
          "df":         DataFrame,     last BQ query result (for charts/tables)
          "sql":        str,           last SQL executed
          "chart":      bool,          True if LLM wants a chart rendered
        }
    """
    async def _run():
        content = types.Content(
            role="user",
            parts=[types.Part(text=user_message)],
        )

        final_text  = ""
        tool_calls  = []
        last_sql    = ""
        last_df     = None

        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=content,
        ):
            # Collect tool call names for display
            if event.content and event.content.parts:
                for part in event.content.parts:
                    # Tool call (function call part)
                    try:
                        if part.function_call and part.function_call.name:
                            tool_calls.append(part.function_call.name)
                            # Capture SQL when run_bq_query is called
                            if part.function_call.name == "run_bq_query":
                                args = dict(part.function_call.args or {})
                                last_sql = args.get("sql", "")
                    except Exception:
                        pass

            # Final response from agent
            if event.is_final_response():
                try:
                    for part in event.content.parts:
                        if part.text:
                            final_text += part.text
                except Exception:
                    pass

        # Re-run the last SQL to get the DataFrame for display
        if last_sql:
            try:
                credentials, _ = google_auth_default()
                client  = bigquery.Client(
                    project=BQ_BILLING_PROJECT,
                    credentials=credentials,
                )
                last_df = client.query(last_sql).result().to_dataframe(
                    create_bqstorage_client=False
                )
                num_cols = last_df.select_dtypes(include="number").columns
                last_df[num_cols] = last_df[num_cols].round(4)
            except Exception:
                pass

        return final_text.strip(), tool_calls, last_df, last_sql

    final_text, tool_calls, last_df, last_sql = asyncio.run(_run())

    chart_requested = "[CHART_REQUESTED]" in final_text
    final_text      = final_text.replace("[CHART_REQUESTED]", "").strip()

    return {
        "text":       final_text,
        "tool_calls": tool_calls,
        "df":         last_df,
        "sql":        last_sql,
        "chart":      chart_requested,
    }

Overwriting agent.py


In [ ]:
# ============================================================
# CELL 5 — Write charts.py
# ============================================================

%%writefile charts.py
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def auto_chart(df: pd.DataFrame, title: str = ""):
    if df is None or df.empty:
        return None

    df        = df.copy()
    num_cols  = df.select_dtypes(include="number").columns.tolist()
    str_cols  = df.select_dtypes(include="object").columns.tolist()
    curr_cols = [c for c in num_cols if c.startswith("current_")]
    prev_cols = [c for c in num_cols if c.startswith("previous_")]

    date_cols = []
    for c in df.columns:
        if "date" in c.lower():
            date_cols.append(c)
        elif c in str_cols:
            try:
                pd.to_datetime(df[c].dropna().head(5))
                date_cols.append(c)
            except Exception:
                pass

    if curr_cols and prev_cols:
        return _comparison_bar(df, curr_cols, prev_cols, str_cols, title)
    if date_cols:
        return _time_series(df, date_cols[0], num_cols, title)
    if str_cols and num_cols:
        return _ranking_bar(df, str_cols[0], num_cols[0], title)
    return None


def _comparison_bar(df, curr_cols, prev_cols, str_cols, title):
    label_col = str_cols[0] if str_cols else None
    pairs = []
    for cc in curr_cols:
        metric = cc.replace("current_", "")
        pc = f"previous_{metric}"
        if pc in prev_cols:
            pairs.append((metric, cc, pc))
    if not pairs:
        return None

    n_cols = min(2, len(pairs))
    n_rows = (len(pairs) + 1) // 2
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[p[0].replace("_", " ").title() for p in pairs],
    )
    for i, (metric, cc, pc) in enumerate(pairs):
        row = i // n_cols + 1
        col = i % n_cols + 1
        x   = df[label_col].astype(str) if label_col else [f"Row {j}" for j in range(len(df))]
        fig.add_trace(go.Bar(name="Current",  x=x, y=df[cc], marker_color="#1f77b4", showlegend=(i==0)), row=row, col=col)
        fig.add_trace(go.Bar(name="Previous", x=x, y=df[pc], marker_color="#aec7e8", showlegend=(i==0)), row=row, col=col)
    fig.update_layout(title_text=title or "Period Comparison", barmode="group",
                      height=350*n_rows, template="plotly_white")
    return fig


def _time_series(df, date_col, num_cols, title):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.sort_values(date_col)
    fig = go.Figure()
    for col in num_cols[:5]:
        fig.add_trace(go.Scatter(x=df[date_col], y=df[col],
                                 mode="lines+markers",
                                 name=col.replace("_", " ").title()))
    fig.update_layout(title_text=title or "Trend Over Time",
                      xaxis_title="Date", template="plotly_white", height=400)
    return fig


def _ranking_bar(df, label_col, value_col, title):
    try:
        df = df.nlargest(15, value_col)
    except Exception:
        df = df.head(15)
    fig = px.bar(df.sort_values(value_col), x=value_col, y=label_col,
                 orientation="h", title=title or f"Top {label_col} by {value_col}",
                 template="plotly_white", color_discrete_sequence=["#1f77b4"])
    fig.update_layout(height=max(300, len(df) * 28), yaxis_title="")
    return fig


Overwriting charts.py


In [ ]:
# ============================================================
# CELL 6 — Write app.py
# ============================================================
%%writefile app.py
import uuid
import streamlit as st
from streamlit_mic_recorder import mic_recorder
from google.cloud import speech
from google.auth import default as google_auth_default
from agent import chat, create_session
from charts import auto_chart
import whisper
import tempfile
import os
from google.api_core.client_options import ClientOptions

st.set_page_config(
    page_title="Brookfield Marketing Assistant",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown("""
<style>
.tool-badge {
    display: inline-block;
    background: #e8f4f8;
    color: #1a6985;
    font-size: 11px;
    padding: 2px 8px;
    border-radius: 10px;
    margin: 2px;
    font-family: monospace;
}
</style>
""", unsafe_allow_html=True)

# ── Session init ───────────────────────────────────────────
if "user_id" not in st.session_state:
    st.session_state.user_id    = str(uuid.uuid4())
    st.session_state.session_id = str(uuid.uuid4())
    create_session(st.session_state.user_id, st.session_state.session_id)

if "messages" not in st.session_state:
    st.session_state.messages = []


# ── STT: audio bytes → text via Google Cloud Speech ───────
def transcribe_audio(audio_bytes: bytes) -> str:
    """
    Transcribes audio using Google Cloud Speech-to-Text.
    Supports multiple languages — auto-detects and always returns English.
    """
    try:
        credentials, _ = google_auth_default()
        client = speech.SpeechClient(
            credentials=credentials,
            client_options=ClientOptions(quota_project_id="generative-insights-poc-bi")
        )
        audio   = speech.RecognitionAudio(content=audio_bytes)
        config  = speech.RecognitionConfig(
            encoding=speech.RecognitionConfig.AudioEncoding.WEBM_OPUS,
            sample_rate_hertz=48000,
            language_code="en-US",            # primary language
            alternative_language_codes=[
                "hi-IN",                       # Hindi
                "ar-SA",                       # Arabic
                "es-ES",                       # Spanish
                "fr-FR",                       # French
                "de-DE",                       # German
                "ja-JP",                       # Japanese
                "ko-KR",                       # Korean
            ],
            enable_automatic_punctuation=True,
        )
        response = client.recognize(config=config, audio=audio)
        if response.results:
            transcript = response.results[0].alternatives[0].transcript
            print(f"Voice transcript: {transcript}")
            return transcript
        return ""
    except Exception as e:
        st.error(f"Speech-to-text error: {e}")
        return ""


# ── Sidebar ────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 📊 Brookfield Marketing Assistant")
    st.markdown("Powered by **Google ADK** + Gemini on Vertex AI")
    st.markdown("---")
    st.markdown("**Try asking:**")
    examples = [
        "How did Search perform last month vs the month before?",
        "Top 10 keywords by internal link clicks in Feb 2026?",
        "Show social spend trend over the last 3 months",
        "Compare Brand vs NonBrand CTR for Q1 2026",
        "Which social campaigns had highest CPM last month?",
        "Show a chart of monthly impressions by channel",
    ]
    for q in examples:
        if st.button(q, use_container_width=True, key=f"ex_{q[:25]}"):
            st.session_state.pending_question = q

    st.markdown("---")
    col1, col2 = st.columns(2)
    with col1:
        if st.button("🗑️ New chat", use_container_width=True):
            import uuid
            from agent import create_session
            st.session_state.messages   = []
            st.session_state.user_id    = str(uuid.uuid4())
            st.session_state.session_id = str(uuid.uuid4())
            create_session(st.session_state.user_id, st.session_state.session_id)
            st.rerun()
    with col2:
        show_sql = st.toggle("Show SQL", value=False)

    st.markdown("---")
    st.markdown("""
    **🎤 Voice input:**
    - Click **Speak** → talk → click **Stop**
    - Works in any browser, no HTTPS required
    - Transcribed via Google Cloud Speech-to-Text
    """)


# ── Chat history ───────────────────────────────────────────
st.title("📊 Brookfield Marketing Analytics")
st.caption("Text or voice input · Conversational analytics powered by Google ADK")

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        if msg["role"] == "user" and msg.get("via_voice"):
            st.caption("🎤 via voice")
        st.markdown(msg["content"])
        if msg["role"] == "assistant":
            if msg.get("tool_calls"):
                badges = "".join(
                    f'<span class="tool-badge">🔧 {t}</span>'
                    for t in msg["tool_calls"]
                )
                st.markdown(badges, unsafe_allow_html=True)
            if show_sql and msg.get("sql"):
                with st.expander("📝 SQL", expanded=False):
                    st.code(msg["sql"], language="sql")
            if msg.get("df") is not None and not msg["df"].empty:
                with st.expander(f"📋 Data ({len(msg['df'])} rows)", expanded=False):
                    st.dataframe(msg["df"], use_container_width=True, hide_index=True)
            if msg.get("fig") is not None:
                st.plotly_chart(msg["fig"], use_container_width=True)


# ── Input area ─────────────────────────────────────────────
# Mic recorder sits above the text input.
# Pipeline: record audio → Google STT → transcript → agent (same as typed input)

pending = st.session_state.pop("pending_question", None)

# Mic recorder widget — returns dict{"bytes": ...} when a recording completes
audio = mic_recorder(
    start_prompt="🎤 Speak",
    stop_prompt="⏹ Stop",
    just_once=True,            # resets after each recording so it doesn't re-fire
    use_container_width=False,
    key="mic_input",
)

# Text input
user_input = st.chat_input("Ask about Search, Social, or Programmatic...")

# Process voice recording if just completed
voice_text = ""
if audio and audio.get("bytes"):
    with st.spinner("🎤 Transcribing..."):
        voice_text = transcribe_audio(audio["bytes"])
    if voice_text:
        st.info(f'🎤 Heard: "{voice_text}"')
    else:
        st.warning("Could not transcribe — please try again or type your question.")

# Priority: typed > voice > example button
question  = user_input or voice_text or pending
via_voice = bool(voice_text and not user_input)

if question:
    with st.chat_message("user"):
        if via_voice:
            st.caption("🎤 via voice")
        st.markdown(question)

    st.session_state.messages.append({
        "role":       "user",
        "content":    question,
        "via_voice":  via_voice,
        "df":         None,
        "sql":        None,
        "tool_calls": [],
        "fig":        None,
    })

    with st.chat_message("assistant"):
        with st.spinner("Agent is thinking..."):
            result = chat(
                user_message=question,
                user_id=st.session_state.user_id,
                session_id=st.session_state.session_id,
            )

        st.markdown(result["text"])

        if result["tool_calls"]:
            badges = "".join(
                f'<span class="tool-badge">🔧 {t}</span>'
                for t in result["tool_calls"]
            )
            st.markdown(badges, unsafe_allow_html=True)

        if show_sql and result["sql"]:
            with st.expander("📝 SQL", expanded=False):
                st.code(result["sql"], language="sql")

        if result["df"] is not None and not result["df"].empty:
            with st.expander(f"📋 Data ({len(result['df'])} rows)", expanded=False):
                st.dataframe(result["df"], use_container_width=True, hide_index=True)

        fig = None
        if result["df"] is not None and not result["df"].empty:
            fig = auto_chart(result["df"], title=question[:60])
            if fig:
                st.plotly_chart(fig, use_container_width=True)

    st.session_state.messages.append({
        "role":       "assistant",
        "content":    result["text"],
        "df":         result["df"],
        "sql":        result["sql"],
        "tool_calls": result["tool_calls"],
        "fig":        fig,
    })


Overwriting app.py


In [ ]:
# ============================================================
# CELL 7 — Download Cloudflare tunnel binary
# ============================================================

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O cloudflared
!chmod +x cloudflared
print("✅ Cloudflare tunnel binary ready")


cloudflared: Text file busy
✅ Cloudflare tunnel binary ready


In [ ]:
# ============================================================
# CELL 8 — Launch Streamlit + Cloudflare Tunnel
# ============================================================
# TWO tunnel options provided — try Option A first.
# If it doesn't work, use Option B (ngrok fallback).
# ─────────────────────────────────────────────────────────────

import subprocess
import time
import threading
subprocess.run(["pkill", "-f", "streamlit"],    capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"],  capture_output=True)
import time
time.sleep(3)
print("✅ Old processes killed — starting fresh")

# Start Streamlit in background
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--browser.gatherUsageStats=false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(4)
print("✅ Streamlit started on port 8501")

# ── Option A: Cloudflare Tunnel (recommended — no account needed) ─

cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("\n⏳ Waiting for Cloudflare tunnel URL...")
cf_url = None
for line in cf_proc.stdout:
    if "trycloudflare.com" in line:
        # Extract URL from the log line
        import re
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            cf_url = match.group(0)
            break

if cf_url:
    print(f"\n✅ Chatbot is live!")
    print(f"🌐 Cloudflare URL: {cf_url}")
    print("\nℹ️  Share this URL with your team.")
    print("ℹ️  URL changes every time you restart — this is normal for free tier.")
    print("ℹ️  Keep this cell running to keep the chatbot alive.")
else:
    print("⚠️  Cloudflare URL not found. Trying ngrok fallback...")

    # ── Option B: ngrok fallback ──────────────────────────────
    try:
        !pip install -q pyngrok
        from pyngrok import ngrok
        public_url = ngrok.connect(8501)
        print(f"\n✅ Chatbot is live (via ngrok)!")
        print(f"🌐 ngrok URL: {public_url}")
    except Exception as e:
        print(f"❌ Both tunnels failed: {e}")
        print("Try running: !./cloudflared tunnel --url http://localhost:8501")

✅ Old processes killed — starting fresh
✅ Streamlit started on port 8501

⏳ Waiting for Cloudflare tunnel URL...

✅ Chatbot is live!
🌐 Cloudflare URL: https://product-collective-heart-fig.trycloudflare.com

ℹ️  Share this URL with your team.
ℹ️  URL changes every time you restart — this is normal for free tier.
ℹ️  Keep this cell running to keep the chatbot alive.
